# Custom Tools and Voice Generation


### Imports

In [ ]:
%pip install -q "crewai[tools]>=1.6.1,<2" python-dotenv openai pydantic exa-py

import os

from crewai.llm import LLM
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
os.makedirs("outputs", exist_ok=True)

---

### Step 1: The @tool Decorator


In [ ]:
import datetime

from crewai.tools import tool


@tool("Create episode filename")
def create_episode_filename(topic: str) -> str:
    """Create a safe filename prefix for a podcast episode topic."""
    safe_topic = topic.lower().replace(" ", "-").replace("/", "-")
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    return f"{safe_topic}-{timestamp}"


print(create_episode_filename.run("AI Agents"))
print(create_episode_filename.description)

### Step 2: Built-In File Tools


In [ ]:
from crewai_tools import EXASearchTool, FileReadTool, FileWriterTool


search_tool = EXASearchTool()
file_writer_tool = FileWriterTool()
file_read_tool = FileReadTool()

print(f"Search: {search_tool.name}")
print(f"Writer: {file_writer_tool.name}")
print(f"Reader: {file_read_tool.name}")

---

### Step 3: Build the Voice Tool (OpenRouter TTS)


In [ ]:
import re
import wave
from pathlib import Path

from openai import OpenAI


VOICE_MAP = {"Joe": "onyx", "Jane": "nova"}
TTS_MODEL = "openai/gpt-4o-mini-tts-2025-12-15"

SPEAKER_RE = re.compile(r"^(Joe|Jane)\s*:\s*(.+)", re.IGNORECASE)


def write_wave_file(
    filename: str,
    pcm: bytes,
    channels: int = 1,
    rate: int = 24000,
    sample_width: int = 2,
) -> None:
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)


def tts_line(text: str, voice: str) -> bytes:
    """Call OpenRouter TTS for a single line, return raw PCM bytes."""
    response = tts_client.audio.speech.create(
        model=TTS_MODEL,
        input=text,
        voice=voice,
        response_format="pcm",
    )
    return response.content


@tool
def voice_tool(script: str) -> str:
    """Generate a multi-speaker podcast audio file from a Joe/Jane script using OpenRouter TTS."""
    all_pcm = bytearray()
    for line in script.strip().splitlines():
        match = SPEAKER_RE.match(line.strip())
        if not match:
            continue
        speaker = match.group(1).title()
        text = match.group(2).strip()
        voice = VOICE_MAP.get(speaker, "onyx")
        all_pcm.extend(tts_line(text, voice))

    filename = (
        outputs_dir / f"podcast-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.wav"
    )
    write_wave_file(str(filename), bytes(all_pcm))
    return str(filename)

### Test the Voice Tool Directly

Run this to confirm the OpenRouter TTS pipeline works before wiring it into an agent. This uses the same `OPENROUTER_API_KEY` as your text agents.


In [ ]:
from IPython.display import Audio, display

sample_script = (
    "Joe: Hi I am Joe, welcome to the Outskill Bootcamp.\n"
    "Jane: Hi Joe, what will we learn today?\n"
    "Joe: Today we will learn about Multi Agent Systems and Orchestration.\n"
    "Jane: Wow that sounds interesting, let's dive in."
)

audio_path = voice_tool.run(sample_script)
display(Audio(audio_path))

---

### Step 4: Structured Research Output


In [ ]:
from pydantic import BaseModel, Field


class ResearchItem(BaseModel):
    title: str = Field(description="Short title for the finding")
    summary: str = Field(description="What happened or why it matters")
    listener_angle: str = Field(description="Why podcast listeners would care")
    source: str = Field(description="Source name or URL when available")


class ResearchReport(BaseModel):
    topic: str = Field(description="Podcast topic being researched")
    episode_angle: str = Field(description="Main angle for the episode")
    items: list[ResearchItem] = Field(description="Research findings for the episode")
    host_questions: list[str] = Field(
        description="Questions hosts can ask during the episode"
    )


def validate_research_report(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if len(report.items) < 5:
        return (False, "The research report must include at least 5 findings.")
    if len(report.host_questions) < 3:
        return (False, "The report must include at least 3 host questions.")
    return (True, report)

---

### Step 5: The Full Voice-Enabled Crew


In [ ]:
from crewai import Agent


researcher = Agent(
    role="{topic} Senior Researcher",
    goal="Find recent releases, articles, topics, and progress about {topic}",
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=12,
    max_execution_time=300,
    respect_context_window=True,
)

reporting_analyst = Agent(
    role="{topic} Reporting Analyst",
    goal="Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You are a meticulous analyst with a keen eye for detail. "
        "You turn complex data into clear podcast briefing reports."
    ),
    llm=llm,
    tools=[file_writer_tool, file_read_tool],
    verbose=True,
    max_iter=4,
    max_rpm=10,
    max_execution_time=240,
    respect_context_window=True,
)

scriptwriter = Agent(
    role="Podcast Scriptwriter",
    goal="Create a fun, natural, interesting podcast script given a report on {topic}",
    backstory=(
        "You are a naturally talented scriptwriter who turns technical reports into engaging, "
        "funny, and articulate podcast dialogue between two hosts."
    ),
    llm=llm,
    tools=[file_writer_tool, file_read_tool, voice_tool],
    verbose=True,
    max_iter=5,
    max_rpm=8,
    max_execution_time=420,
    respect_context_window=True,
)

In [ ]:
from typing import Any, Tuple

from crewai import Task
from crewai.tasks.task_output import TaskOutput


research_task = Task(
    description=(
        "Conduct thorough podcast research about {topic}. Find interesting and relevant information. "
        "The current date is {current_month} {current_year}."
    ),
    expected_output="A structured research report with at least 5 findings and 3 host questions.",
    output_pydantic=ResearchReport,
    guardrail=validate_research_report,
    agent=researcher,
)

reporting_task = Task(
    description=(
        "Review the research context and organize each topic into a full section for a report. "
        "Make the report detailed and useful for a podcast production team."
    ),
    expected_output="A markdown report with episode angle, sections, facts, host questions, and transitions.",
    guardrail=(
        "The report must include at least 3 clearly labeled sections with headers. "
        "Each section must contain at least 2 bullet points of useful facts."
    ),
    agent=reporting_analyst,
    context=[research_task],
    output_file="outputs/podcast_report.md",
)

# Prompt Engineering: Chain-of-Thought — sequential reasoning steps guide the agent before it writes
scripting_task = Task(
    description=(
        "Review the report and write an elaborate podcast script between two hosts. "
        "First, identify the 3-4 most compelling moments or surprising reveals — these become the episode peaks. "
        "Then plan a narrative arc: cold open hook, topic order, and a closing takeaway. "
        "Finally, write the full dialogue between Joe and Jane following that arc. "
        "The only speakers are Joe and Jane. Use speaker labels exactly as Joe: and Jane:. "
        "Each dialogue line must contain only spoken words — no parenthetical stage directions like (laughs) or (whispers), "
        "and no bracketed sound cues like [music] or [pause]. The script will be fed directly to a TTS engine. "
        "Write the full script to the output file."
    ),
    expected_output=(
        "A complete Joe/Jane podcast script with at least 12 lines of clean spoken dialogue. "
        "Every line starts with Joe: or Jane: followed only by the words they speak."
    ),
    guardrail=(
        "The script must only use Joe and Jane as speakers. "
        "No line may contain parenthetical stage directions or bracketed sound cues. "
        "It must include at least 12 lines of dialogue."
    ),
    agent=scriptwriter,
    context=[reporting_task],
    output_file="outputs/podcast_script.md",
)

---

## Step 6: Run the End-to-End Podcaster

```text
Topic -> Researcher -> Reporting Analyst -> Scriptwriter -> Script file
```

After the crew finishes, we run `voice_tool` on the saved script as a deterministic post-processing step — no guessing whether the agent will call it.


In [ ]:
from crewai import Crew, Process


voice_podcaster_crew = Crew(
    agents=[researcher, reporting_analyst, scriptwriter],
    tasks=[research_task, reporting_task, scripting_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=30,
)

inputs = {
    "topic": "Building a Career in AI Agent Engineering",
    "current_month": datetime.datetime.now().strftime("%B"),
    "current_year": str(datetime.datetime.now().year),
}

final_result = voice_podcaster_crew.kickoff(inputs=inputs)
print(final_result)

In [ ]:
script_path = Path("outputs/podcast_script.md")
script_text = script_path.read_text()
print(f"Script loaded: {len(script_text)} chars, generating audio...")

final_audio_path = voice_tool.run(script_text)
print(f"Audio saved: {final_audio_path}")

In [ ]:
display(Audio(final_audio_path))

---

## Step 7: Reuse With Any Topic

Once the crew is defined, changing the episode topic is just changing the inputs.


In [ ]:
new_inputs = {
    "topic": "AI agents for software engineering",
    "current_month": datetime.datetime.now().strftime("%B"),
    "current_year": str(datetime.datetime.now().year),
}

# Uncomment to create another episode.
# voice_podcaster_crew.kickoff(inputs=new_inputs)